# 07 · Validasi & ablation

**Tujuan.** Menguji perubahan pada decoder **terhadap acuan** (`06` v2) dengan protokol yang sama untuk semua:
proxy tanpa GT di seluruh match, ditambah GT publik sebagai informasi. Notebook ini **tidak menulis submission**;
ia mencatat apa yang dicoba, hasilnya, dan keputusan diadopsi/ditolak — termasuk hasil negatif.

| | |
|---|---|
| **Input** | `cache/dead.npz`, `cache/possession_model.npz`, `cache/features_ha.npz`, `cache/restarts.csv` |
| **Output** | laporan `cache/reports/07_validasi_ablation.*` |
| **Waktu** | ± 1–2 menit |

### Aturan membaca tabel

- **Proxy tanpa GT** (dari `possession_proxies`, `common.ipynb` §9) adalah dasar keputusan. Yang terpenting:
  `didukung sentuhan %` (pergantian yang masuk akal secara fisik), `restart cocok %`, dan jumlah possession.
- **GT publik** hanya informasi: 17 possession, satu possession ≈ ± 6 poin spellF1. Selisih < ± 0,05 tidak berarti.
- Perubahan diadopsi hanya bila proxy membaik **dan** GT publik tidak memburuk; perubahan yang netral ditolak
  (kompleksitas tanpa bukti).

### Eksperimen langkah 2.7 (pergantian & batas)

| | Hipotesis | Bagian |
|---|---|---|
| A | Momen **kontrol jelas** sebagai jangkar menangkap possession pendek yang terlewat | 3 |
| B | **Batas digeser** ke sentuhan pertama / kontrol bebas pertama tim baru membuat batas lebih tepat | 4 |
| C | Bukti bentuk tim **tertinggal waktu**; menggesernya memperbaiki pergantian | 5 |

In [ ]:
%run ./common.ipynb

## Parameter

Sama dengan `06` v2 agar acuan identik, ditambah parameter eksperimen.

In [ ]:
SWITCH_S = 1.0
L_CLIP = 6.0
CTRL_VARIANTS = [(1.5, 3.0, 12), (1.0, 2.0, 8)]      # (radius sendiri, radius lawan, frame minimum)
CTRL_K = [0.5, 1.0, 2.0]                             # kekuatan jangkar: k × biaya pergantian per momen kontrol penuh
LAGS_S = [-4, -2, -1, 0, 1, 2, 3, 4, 6]

In [ ]:
rep = Report("07_validasi_ablation")
with np.load(CACHE / "dead.npz") as dz:
    d_mask = dz["d_mask"]
with np.load(CACHE / "possession_model.npz") as mz:
    L_total, L_shape, L_near, pseudo = (mz["L_total"].astype(float), mz["L_shape"].astype(float),
                                        mz["L_near"].astype(float), mz["pseudo"])
with np.load(CACHE / "features_ha.npz") as fz:
    d_H, d_A, near_share = fz["d_H"], fz["d_A"], fz["near_share"].astype(float)
observed = load_prep()["ball_src"] == 1
restarts = pd.read_csv(CACHE / "restarts.csv", keep_default_na=False)
live = GRADED & ~d_mask
L0 = np.clip(L_total, -L_CLIP, L_CLIP)
L_MED = float(np.median(np.abs(L0[live])))
COST = SWITCH_S * FPS * L_MED
dH0, dA0 = np.nan_to_num(d_H, nan=99.0), np.nan_to_num(d_A, nan=99.0)
ROWS = []


def evaluate(group, name, lab, note=""):
    pub = score_labels_public(lab)
    row = {"kelompok": group, "varian": name, **possession_proxies(lab, live, pseudo, restarts, d_H, d_A, observed),
           "publik spellF1": round(pub["spell_f1_mean"], 3), "publik macroF1": round(pub["macro_f1"], 3),
           "publik possession": int(pub["n_pred_spells"]), "catatan": note}
    ROWS.append(row)
    return row


def show(group):
    t = pd.DataFrame([r for r in ROWS if r["kelompok"] == group]).drop(columns="kelompok").set_index("varian")
    rep.table(t, "\n[{}]".format(group))
    return t

## 1. Acuan: `06` v2

In [ ]:
lab_ref = decode_possession(L0, d_mask, COST)
evaluate("acuan", "v2", lab_ref)
show("acuan")

## 2. Diagnosis per possession di GT publik

Untuk setiap possession GT: IoU terbaik dengan possession prediksi tim yang sama, dan selisih awal/akhir (s).
Untuk possession pendek (< 8 s): seberapa sering tiap jenis bukti menunjuk tim yang benar, dan berapa frame
kontrol jelas yang dimiliki tim itu. Ini menjelaskan **mengapa** possession terlewat — tidak dipakai untuk menyetel.

In [ ]:
gt_codes = pm.label_codes(pm.read_label_csv(str(GT_PUBLIC)))
runs_gt = pm.valid_runs(gt_codes)
gt_sp = pm.possession_spells(gt_codes, runs_gt)
pr_sp = pm.possession_spells(pm.label_codes(np.asarray(lab_ref, dtype=object)), runs_gt)


def ctrl_mask(d_own, d_oth, r_own, r_oth, n_min):
    cond = live & observed & (d_own <= r_own) & (d_oth >= r_oth)
    out = np.zeros(N_FRAMES, dtype=bool)
    for a, b in zip(*runs(cond)):
        if b - a >= n_min:
            out[a:b] = True
    return out


cH0, cA0 = ctrl_mask(dH0, dA0, 1.5, 3.0, 12), ctrl_mask(dA0, dH0, 1.5, 3.0, 12)
diag = []
for a, b, c in gt_sp:
    best, oa, ob = 0.0, np.nan, np.nan
    for x, y, cc in pr_sp:
        inter = max(0, min(b, y) - max(a, x))
        if cc == c and inter:
            iou = inter / ((b - a) + (y - x) - inter)
            if iou > best:
                best, oa, ob = iou, (x - a) / FPS, (y - b) / FPS
    sgn = 1 if c == pm.H else -1
    vis = L_near[a:b] != 0
    diag.append({"durasi (s)": round((b - a) / FPS, 1), "tim": "H" if c == pm.H else "A", "IoU": round(best, 2),
                 "awal (s)": oa, "akhir (s)": ob,
                 "bentuk benar %": round(100 * float((np.sign(L_shape[a:b]) == sgn).mean())),
                 "terdekat benar %": round(100 * float((np.sign(L_near[a:b][vis]) == sgn).mean())) if vis.any() else np.nan,
                 "frame kontrol jelas": int((cH0 if c == pm.H else cA0)[a:b].sum())})
diag = pd.DataFrame(diag)
rep.table(diag, "possession GT publik vs v2:")
rep("possession GT dengan IoU < 0,3: {} dari {} (durasi: {})".format(
    int((diag.IoU < 0.3).sum()), len(diag), diag.loc[diag.IoU < 0.3, "durasi (s)"].tolist()))

## 3. Eksperimen A — jangkar kontrol jelas

Pada frame "kontrol jelas" tim T (pemain T ≤ r₁ dari bola teramati, lawan ≥ r₂, bertahan ≥ n frame) log-odds
ditambah ± `k × biaya / n`, sehingga satu momen kontrol penuh membayar `k` kali biaya pergantian.

In [ ]:
for r1, r2, n_min in CTRL_VARIANTS:
    cH, cA = ctrl_mask(dH0, dA0, r1, r2, n_min), ctrl_mask(dA0, dH0, r1, r2, n_min)
    for k in CTRL_K:
        L = L0 + (k * COST / n_min) * (cH.astype(float) - cA)
        evaluate("A: jangkar kontrol", "r={}/{} n={} k={}".format(r1, r2, n_min, k), decode_possession(L, d_mask, COST))
tab_a = show("A: jangkar kontrol")

## 4. Eksperimen B — menggeser batas pergantian

Setelah decoding, setiap batas H↔A dicari ulang dalam jendela [t − mundur, t + maju] (tidak melewati possession
tetangga):

- **sentuhan pertama**: frame pertama pemain tim baru ≤ 2 m dan terdekat, yang lalu bertahan terdekat ≥ 60% dari 12 frame;
- **kontrol bebas pertama**: frame pertama pemain tim baru ≤ 1,5 m dan lawan ≥ 3 m selama `hold` frame.

Kolom catatan: median geseran dan berapa batas maju/mundur.

In [ ]:
def refine(lab, mode, back_s=3.0, fwd_s=3.0, hold=6):
    lab = lab.copy()
    t_idx = np.flatnonzero((lab[1:] != lab[:-1]) & np.isin(lab[1:], ["H", "A"]) & np.isin(lab[:-1], ["H", "A"])) + 1
    moved = []
    for t in t_idx:
        new, old = lab[t], lab[t - 1]
        d_new, d_old = (dH0, dA0) if new == "H" else (dA0, dH0)
        ps = t - 1
        while ps > 0 and lab[ps - 1] == old:
            ps -= 1
        ne = t
        while ne < N_FRAMES - 1 and lab[ne + 1] == new:
            ne += 1
        lo, hi = max(int(t - back_s * FPS), ps + 1), min(int(t + fwd_s * FPS), ne)
        best = None
        for g in range(lo, hi):
            if mode == "touch":
                if observed[g] and d_new[g] <= 2.0 and d_new[g] < d_old[g] \
                        and ((d_new[g:g + 12] < d_old[g:g + 12]) & observed[g:g + 12]).sum() >= 0.6 * 12:
                    best = g
                    break
            else:
                if (observed[g:g + hold] & (d_new[g:g + hold] <= 1.5) & (d_old[g:g + hold] >= 3.0)).all():
                    best = g
                    break
        if best is not None and best != t:
            if best < t:
                lab[best:t] = new
            else:
                lab[t:best] = old
            moved.append((best - t) / FPS)
    note = "geser median {:+.2f} s; maju {} / mundur {}".format(
        float(np.median(moved)) if moved else 0.0, sum(m < 0 for m in moved), sum(m > 0 for m in moved))
    return lab, note


for mode, kw in [("touch", dict(fwd_s=1.0)), ("free", dict()), ("free", dict(back_s=2.0, fwd_s=2.0)),
                 ("free", dict(hold=12))]:
    lab_r, note = refine(lab_ref, mode, **kw)
    evaluate("B: geser batas", "{} {}".format(mode, kw or ""), lab_r, note)
tab_b = show("B: geser batas")

## 5. Eksperimen C — keterlambatan bukti bentuk tim

**Tanpa GT:** korelasi antara porsi tim terdekat ± 1 s (`near_share`, sesaat) dan log-odds bentuk tim yang digeser
τ detik, di frame hidup seluruh match. Puncak di τ > 0 berarti bentuk tim **tertinggal**. Lalu decoder dijalankan
dengan bukti bentuk yang digeser.

In [ ]:
ok0 = np.flatnonzero(live & np.isfinite(near_share))
xc = []
for tau_s in LAGS_S:
    tau = int(tau_s * FPS)
    idx = ok0[(ok0 + tau >= 0) & (ok0 + tau < N_FRAMES)]
    idx = idx[live[idx + tau]]
    xc.append({"τ (s)": tau_s, "korelasi": round(float(np.corrcoef(near_share[idx], L_shape[idx + tau])[0, 1]), 4)})
xc = pd.DataFrame(xc).set_index("τ (s)")
rep.table(xc, "korelasi near_share[t] vs L_shape[t+τ]:")
rep("puncak korelasi pada τ = {} s".format(xc.korelasi.idxmax()))

for tau_s in (1, 2, 3):
    tau = int(tau_s * FPS)
    shifted = np.zeros(N_FRAMES)
    shifted[:N_FRAMES - tau] = L_shape[tau:]
    evaluate("C: geser bentuk tim", "bentuk tim dari t+{} s".format(tau_s),
             decode_possession(np.clip(shifted + L_near, -L_CLIP, L_CLIP), d_mask, COST))
tab_c = show("C: geser bentuk tim")

In [ ]:
all_rows = pd.DataFrame(ROWS)
rep.save({"rows": all_rows.to_dict("records"), "diagnosis": diag.to_dict("records"),
          "xcorr": xc.reset_index().to_dict("records")})

## Ringkasan & keputusan (langkah 2.7)

| Eksperimen | Hasil | Keputusan |
|---|---|---|
| A — jangkar kontrol jelas | k = 0,5: proxy fisik & publik **tidak berubah**; k ≥ 1: possession bertambah (224–270), tetapi `didukung sentuhan` turun dan publik tidak membaik. Kenaikan `pseudo-label cocok` bersifat melingkar (jangkar = pseudo-label) | **ditolak** (netral → tidak menambah kompleksitas) |
| B — geser batas | `didukung sentuhan` naik **dengan sendirinya** (batas sengaja dipindah ke sentuhan); publik turun di semua varian. Geseran "kontrol bebas" simetris (maju ≈ mundur) → batas v2 **tidak bias sistematis** | **ditolak** |
| C — keterlambatan bentuk tim | **Fakta baru tanpa GT:** bukti bentuk tim tertinggal ± 4 s dari bukti bola (puncak korelasi). Tetapi menggesernya merusak `didukung sentuhan` dan publik — bentuk tim tetap paling berguna sebagai bukti *lambat* yang menstabilkan, bukan penentu waktu | **ditolak**; dicatat untuk laporan |

**Mengapa possession publik terlewat (bagian 2)**

- 6 dari 17 possession GT tidak tertangkap (IoU < 0,3). Tiga di antaranya berdurasi **≤ 1,3 s tanpa bukti apa pun**
  (bentuk tim 0%, pemain terdekat ≤ 42%, tanpa kontrol jelas) — kemungkinan sentuhan/defleksi singkat yang dicatat
  tracking terkoreksi. Ini batas yang realistis untuk pendekatan berbasis tracking otomatis; mengejarnya akan
  menciptakan banyak possession palsu di seluruh match.
- Sisanya possession pendek 3–8 s di mana bukti bentuk tim tertinggal (fakta C) dan bukti bola terlalu lemah.
- Possession panjang 42,4 s **berakhir 18 s terlalu awal**: dipotong `D` palsu yang tersisa → kembali ke detektor
  bola mati.

**Keputusan:** decoder H/A **v2 dipertahankan**. Upaya berikutnya dialihkan ke sumber kesalahan yang lebih besar:
`D` palsu/terlewat yang tersisa (presisi/recall publik ± 0,80) dan tim restart (Fase 3.2).